# 12 Final test results (Step 19)

The test block (the last 10% by time, 59,054 rows) is scored **once**, after the group has confirmed the experiment
freeze. This notebook trains nothing: it reads the test rows of `results/log.csv` and the saved test scores in
`results/preds/`, and builds the tables for the report.

**How the test rows are made** (`src/final_test.py`, one model per process, each person runs their own):

```
.venv/bin/python -u -m src.final_test --model catboost --freeze-confirmed
```

For each of the model's 12 cells (G0–G3 × seeds 0, 1, 2), the settings validated in Step 16 are refitted on the
train block exactly as in the validation run, the refit must reproduce the logged validation PR-AUC, and the test
block is scored with the threshold stored at validation. Nothing is tuned or re-thresholded on the test block.

The **segment blend** (G3 only; fixed before the freeze in `src/ensemble.py`: known customers → XGBoost, unknown →
CatBoost 0.6, LightGBM 0.2, XGBoost 0.1, random forest 0.1) needs no training. Once the four input models have their
G3 test rows, `python -m src.final_test --blend --freeze-confirmed` combines their saved test scores and applies the
blend's stored validation threshold.

A cell is shown only when all three seeds are in the log; section 1 lists what is missing.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src import ensemble, experiment, feature_sets as fs, final_test, segments
from src.metrics import top_rows
from src.split import holdout_indices

MODELS = list(final_test.TABLE_MODELS)          # six models plus the segment blend (G3 only)
BLEND = ensemble.BLEND_NAME
LABELS = final_test.NAMES
FEATURE_SETS = list(fs.FEATURE_SETS)
SEEDS = list(experiment.VAL_SEEDS)
N_BOOT = 200
METRICS = {"pr_auc": "PR-AUC", "auroc": "AUROC", "recall_top1pct": "recall in top 1%",
           "precision_top1pct": "precision in top 1%", "precision": "precision at stored threshold",
           "recall": "recall at stored threshold"}
pd.set_option("display.width", 160)

## 1. What has been run

In [2]:
spec = fs.load_spec()
ids = pd.read_parquet(ROOT / spec["parquet"],
                      columns=["TransactionID", "TransactionDT", "TransactionDay", "uid", "isFraud"])
train_idx, val_idx, test_idx = holdout_indices(ids)

log = experiment.current_runs(pd.read_csv(experiment.LOG_PATH))
test = log[(log["split"] == "test") & log["model"].isin(MODELS) & log["feature_set"].isin(FEATURE_SETS)].copy()
val = log[(log["split"] == "val") & (log["cv_scheme"] == "holdout") & log["model"].isin(MODELS)
          & log["feature_set"].isin(FEATURE_SETS)].copy()

seeds_done = test.groupby(["model", "feature_set"])["seed"].apply(lambda s: sorted(set(s)))
status = {(m, g): "n/a" if m == BLEND and g != ensemble.BLEND_SET else ("done" if seeds_done.get((m, g), []) == SEEDS else
                   f"{len(seeds_done.get((m, g), []))}/{len(SEEDS)} seeds" if (m, g) in seeds_done else "not run")
          for m in MODELS for g in FEATURE_SETS}
display(pd.DataFrame([[status[(m, g)] for g in FEATURE_SETS] for m in MODELS],
                     index=[LABELS[m] for m in MODELS], columns=FEATURE_SETS))
print(f"Test rows in the log: {len(test)} of {final_test.EXPECTED_TEST_ROWS}.")
# Nothing about the test block (labels, segment shares) is read until test rows exist, i.e. after the freeze.
FROZEN_RUN = len(test) > 0
n_test = len(test_idx)
fraud_test = int(ids["isFraud"].iloc[test_idx].sum()) if FROZEN_RUN else None
if FROZEN_RUN:
    print(f"test block: {n_test:,} rows, {fraud_test:,} fraud; the top 1% is {top_rows(n_test):,} rows.")
done = test[np.array([status[(m, g)] == "done" for m, g in zip(test["model"], test["feature_set"])], dtype=bool)].copy()
if FROZEN_RUN:
    done["precision_top1pct"] = done["recall_top1pct"] * fraud_test / top_rows(n_test)

,G0,G1,G2,G3
Logistic regression,done,done,done,done
Random forest,done,done,done,done
LightGBM,done,done,done,done
XGBoost,done,done,done,done
CatBoost,done,done,done,done
MLP,done,done,done,done
Segment blend (G3),n/a,n/a,n/a,done


Test rows in the log: 75 of 75.
test block: 59,054 rows, 2,213 fraud; the top 1% is 591 rows.


## 2. Test results (mean ± sample sd over seeds 0, 1, 2)

Precision in the top 1% is derived from the logged recall in the top 1%, as in `10_model_comparison`. Precision and
recall at the threshold use the threshold stored at validation, so unlike the validation tables they are not
optimistic.

In [3]:
def table(frame, metric):
    g = frame.groupby(["model", "feature_set"])[metric]
    cell = g.mean().map("{:.4f}".format) + " ± " + g.std().map("{:.4f}".format)
    out = cell.unstack().reindex(index=MODELS, columns=FEATURE_SETS)
    return out.rename(index=LABELS).fillna("—")

if done.empty:
    print("No complete test cells yet: run src.final_test for each model after the experiment freeze.")
else:
    for metric, label in METRICS.items():
        print(label)
        display(table(done, metric))

PR-AUC


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,0.1572 ± 0.0000,0.1631 ± 0.0000,0.1656 ± 0.0000,0.1681 ± 0.0000
Random forest,0.5295 ± 0.0021,0.5322 ± 0.0008,0.5348 ± 0.0018,0.5332 ± 0.0030
LightGBM,0.5657 ± 0.0016,0.5807 ± 0.0051,0.5791 ± 0.0026,0.5753 ± 0.0035
XGBoost,0.5270 ± 0.0236,0.5711 ± 0.0023,0.5728 ± 0.0009,0.5700 ± 0.0007
CatBoost,0.5459 ± 0.0027,0.5429 ± 0.0031,0.5417 ± 0.0033,0.5359 ± 0.0033
MLP,0.4554 ± 0.0147,0.4718 ± 0.0036,0.4594 ± 0.0043,0.4610 ± 0.0062
Segment blend (G3),—,—,—,0.6100 ± 0.0003


AUROC


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,0.8322 ± 0.0000,0.8397 ± 0.0000,0.8406 ± 0.0000,0.8410 ± 0.0000
Random forest,0.9035 ± 0.0009,0.9070 ± 0.0010,0.9111 ± 0.0016,0.9100 ± 0.0012
LightGBM,0.9166 ± 0.0008,0.9205 ± 0.0014,0.9193 ± 0.0010,0.9192 ± 0.0005
XGBoost,0.9027 ± 0.0064,0.9131 ± 0.0006,0.9159 ± 0.0016,0.9155 ± 0.0007
CatBoost,0.9061 ± 0.0019,0.9070 ± 0.0015,0.9085 ± 0.0016,0.9066 ± 0.0012
MLP,0.8603 ± 0.0132,0.8704 ± 0.0041,0.8594 ± 0.0063,0.8598 ± 0.0049
Segment blend (G3),—,—,—,0.9290 ± 0.0010


recall in top 1%


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,0.0062 ± 0.0000,0.0068 ± 0.0000,0.0073 ± 0.0000,0.0078 ± 0.0000
Random forest,0.2282 ± 0.0008,0.2291 ± 0.0016,0.2299 ± 0.0011,0.2282 ± 0.0008
LightGBM,0.2395 ± 0.0016,0.2402 ± 0.0005,0.2396 ± 0.0022,0.2393 ± 0.0010
XGBoost,0.2317 ± 0.0030,0.2389 ± 0.0007,0.2369 ± 0.0009,0.2381 ± 0.0012
CatBoost,0.2336 ± 0.0005,0.2339 ± 0.0009,0.2332 ± 0.0009,0.2339 ± 0.0007
MLP,0.2210 ± 0.0041,0.2244 ± 0.0011,0.2241 ± 0.0020,0.2283 ± 0.0039
Segment blend (G3),—,—,—,0.2404 ± 0.0005


... (2 outputs left out)

precision at stored threshold


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,0.2461 ± 0.0000,0.2459 ± 0.0000,0.2539 ± 0.0000,0.2602 ± 0.0000
Random forest,0.5500 ± 0.0284,0.5810 ± 0.0474,0.5586 ± 0.0157,0.5833 ± 0.0294
LightGBM,0.6598 ± 0.0292,0.6601 ± 0.0166,0.7068 ± 0.0085,0.6870 ± 0.0091
XGBoost,0.6702 ± 0.0142,0.6675 ± 0.0031,0.6837 ± 0.0126,0.6846 ± 0.0426
CatBoost,0.7119 ± 0.0223,0.6560 ± 0.0285,0.6422 ± 0.0393,0.5947 ± 0.0305
MLP,0.5907 ± 0.0535,0.5908 ± 0.0339,0.6014 ± 0.0208,0.5383 ± 0.0584
Segment blend (G3),—,—,—,0.7330 ± 0.0266


recall at stored threshold


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,0.3412 ± 0.0000,0.3687 ± 0.0000,0.3696 ± 0.0000,0.3267 ± 0.0000
Random forest,0.4826 ± 0.0219,0.4840 ± 0.0206,0.4906 ± 0.0098,0.4755 ± 0.0140
LightGBM,0.4617 ± 0.0194,0.4769 ± 0.0150,0.4511 ± 0.0130,0.4483 ± 0.0136
XGBoost,0.4222 ± 0.0302,0.4718 ± 0.0027,0.4638 ± 0.0106,0.4636 ± 0.0293
CatBoost,0.4222 ± 0.0064,0.4472 ± 0.0117,0.4508 ± 0.0151,0.4727 ± 0.0204
MLP,0.3776 ± 0.0221,0.3942 ± 0.0217,0.3717 ± 0.0125,0.4032 ± 0.0332
Segment blend (G3),—,—,—,0.4841 ± 0.0128


## 3. Validation to test

Mean PR-AUC on validation and on test for each cell, and the change. A drop from validation to test is expected
(later months, and validation was used to choose the settings and the threshold); what matters is whether the
ranking of models and feature sets holds.

In [4]:
if not done.empty:
    t_mean = done.groupby(["model", "feature_set"])["pr_auc"].mean()
    v_mean = val.groupby(["model", "feature_set"])["pr_auc"].mean()
    change = (t_mean - v_mean.reindex(t_mean.index)).unstack().reindex(index=MODELS, columns=FEATURE_SETS)
    print("change in mean PR-AUC, test − validation")
    display(change.rename(index=LABELS).round(4))
    rank = pd.DataFrame({"validation": v_mean.reindex(t_mean.index), "test": t_mean})
    rank["validation rank"] = rank["validation"].rank(ascending=False).astype(int)
    rank["test rank"] = rank["test"].rank(ascending=False).astype(int)
    print("cells ordered by test PR-AUC")
    display(rank.sort_values("test", ascending=False).rename(index=LABELS, level=0).round(4))

change in mean PR-AUC, test − validation


feature_set,G0,G1,G2,G3
model,,,,
Logistic regression,-0.1823,-0.1891,-0.1850,-0.1844
Random forest,-0.0116,-0.0285,-0.0319,-0.0288
LightGBM,-0.0420,-0.0414,-0.0447,-0.0487
XGBoost,-0.0787,-0.0783,-0.0721,-0.0734
CatBoost,-0.0039,-0.0196,-0.0086,-0.0125
MLP,0.0317,0.0315,0.0234,0.0231
Segment blend (G3),NaN,NaN,NaN,-0.0451


cells ordered by test PR-AUC


validation    test  validation rank  test rank
model               feature_set                                                
Segment blend (G3)  G3               0.6551  0.6100                1          1
LightGBM            G1               0.6221  0.5807                7          2
                    G2               0.6238  0.5791                6          3
                    G3               0.6240  0.5753                5          4
XGBoost             G2               0.6449  0.5728                3          5
                    G1               0.6495  0.5711                2          6
                    G3               0.6434  0.5700                4          7
LightGBM            G0               0.6076  0.5657                8          8
CatBoost            G0               0.5498  0.5459               15          9
                    G1               0.5625  0.5429               11         10
                    G2               0.5504  0.5417               14         11
                    G3               0.5485  0.5359               16         12
Random forest       G2               0.5667  0.5348               10         13
                    G3               0.5620  0.5332               12         14
                    G1               0.5607  0.5322               13         15
                    G0               0.5411  0.5295               17         16
XGBoost             G0               0.6057  0.5270                9         17
MLP                 G1               0.4404  0.4718               18         18
                    G3               0.4379  0.4610               19         19
                    G2               0.4360  0.4594               20         20
                    G0               0.4237  0.4554               21         21
Logistic regression G3               0.3525  0.1681               22         22
                    G2               0.3506  0.1656               24         23
                    G1               0.3521  0.1631               23         24
                    G0               0.3395  0.1572               25         25

## 4. Known vs unknown customers (test block)

A test row is **known** if its `uid` appears in the train block, the rows every scored model was fitted on.
PR-AUC is the mean over seeds with a 95% bootstrap interval over rows (`N_BOOT` resamples), from the saved test
scores.

In [5]:
def known_unknown():
    known_by_id = pd.Series(segments.known_uid(ids["uid"], set(ids["uid"].iloc[train_idx])),
                            index=ids["TransactionID"].to_numpy())
    k_test = known_by_id.iloc[test_idx]
    print(f"test rows: known {k_test.mean():.1%}, unknown {1 - k_test.mean():.1%}")
    rows = []
    for m in MODELS:
        for g in FEATURE_SETS:
            paths = [ensemble.blend_preds_path(experiment.PREDS_DIR, "test", s) if m == BLEND
                 else final_test.preds_path(experiment.PREDS_DIR, m, g, s) for s in SEEDS]
            if status[(m, g)] != "done" or not all(p.exists() for p in paths):
                continue
            preds = [pd.read_parquet(p) for p in paths]
            if not all(p["TransactionID"].equals(preds[0]["TransactionID"]) for p in preds):
                raise ValueError(f"{m}/{g}: saved test scores cover different rows")
            known = known_by_id.reindex(preds[0]["TransactionID"]).to_numpy()
            t = segments.segment_pr_auc(preds[0]["isFraud"].to_numpy(), [p["score"].to_numpy() for p in preds],
                                        known, n_boot=N_BOOT)
            rows.append(t.assign(model=m, feature_set=g))
    if not rows:
        print("No saved test scores for complete cells yet.")
        return
    seg = pd.concat(rows, ignore_index=True)
    display(seg.groupby("segment")[["rows", "fraud"]].first())
    seg["cell"] = seg.apply(lambda r: f"{r.pr_auc:.4f} [{r.low:.3f}, {r.high:.3f}]", axis=1)
    display(seg.pivot_table(index="model", columns=["feature_set", "segment"], values="cell", aggfunc="first")
            .reindex(index=[m for m in MODELS if m in set(seg["model"])]).rename(index=LABELS))

if FROZEN_RUN:
    known_unknown()
else:
    print("No test rows yet; the test block is not read before the experiment freeze.")

test rows: known 38.4%, unknown 61.6%


,rows,fraud
segment,,
known,22695,385
unknown,36359,1828


feature_set                             G0                                            G1                                            G2                         \
segment                              known                unknown                  known                unknown                  known                unknown   
model                                                                                                                                                           
Logistic regression  0.0401 [0.036, 0.046]  0.4154 [0.394, 0.438]  0.0389 [0.035, 0.045]  0.4369 [0.415, 0.459]  0.0390 [0.035, 0.045]  0.4449 [0.424, 0.469]   
Random forest        0.4275 [0.379, 0.480]  0.5597 [0.537, 0.584]  0.5114 [0.459, 0.561]  0.5502 [0.525, 0.577]  0.5418 [0.490, 0.591]  0.5483 [0.523, 0.575]   
LightGBM             0.6685 [0.622, 0.711]  0.5565 [0.535, 0.579]  0.7271 [0.679, 0.765]  0.5621 [0.541, 0.585]  0.7382 [0.698, 0.777]  0.5581 [0.536, 0.581]   
XGBoost              0.7221 [0.673, 0.759]  0.5027 [0.477, 0.526]  0.8046 [0.764, 0.835]  0.5265 [0.503, 0.550]  0.8172 [0.783, 0.847]  0.5275 [0.503, 0.549]   
CatBoost             0.4591 [0.411, 0.508]  0.5691 [0.545, 0.592]  0.4532 [0.413, 0.502]  0.5650 [0.541, 0.589]  0.4597 [0.419, 0.506]  0.5628 [0.539, 0.586]   
MLP                  0.2275 [0.191, 0.269]  0.5016 [0.478, 0.524]  0.2639 [0.227, 0.311]  0.5136 [0.494, 0.536]  0.2794 [0.239, 0.324]  0.4964 [0.475, 0.518]   
Segment blend (G3)                     NaN                    NaN                    NaN                    NaN                    NaN                    NaN   

feature_set                             G3                         
segment                              known                unknown  
model                                                              
Logistic regression  0.0380 [0.034, 0.044]  0.4465 [0.425, 0.470]  
Random forest        0.5222 [0.464, 0.567]  0.5500 [0.527, 0.577]  
LightGBM             0.7461 [0.707, 0.782]  0.5535 [0.531, 0.576]  
XGBoost              0.7973 [0.761, 0.828]  0.5288 [0.504, 0.551]  
CatBoost             0.4141 [0.374, 0.462]  0.5629 [0.540, 0.586]  
MLP                  0.2270 [0.191, 0.270]  0.5078 [0.486, 0.531]  
Segment blend (G3)   0.7973 [0.761, 0.828]  0.5742 [0.551, 0.597]

## 5. Segment blend vs XGBoost (G3)

The blend against its strongest input on the same feature set, seed by seed, on validation and test; and against
XGBoost on G1, the best single cell on validation. Known vs unknown for both is in section 4.

In [6]:
cmp = log[(log["split"].isin(["val", "test"])) & (log["cv_scheme"] == "holdout")
          & (((log["model"] == BLEND) & (log["feature_set"] == "G3"))
             | ((log["model"] == "xgboost") & log["feature_set"].isin(["G3", "G1"])))].copy()
cmp["cell"] = cmp["model"].map(LABELS) + " " + cmp["feature_set"]
if (cmp["split"] == "test").any():
    tab = cmp.pivot_table(index="cell", columns=["split", "seed"], values="pr_auc")
    tab[("test", "mean")] = tab["test"].mean(axis=1)
    tab[("val", "mean")] = tab["val"].mean(axis=1)
    display(tab.round(4))
else:
    print("No test rows yet.")

split                    test                     val                    test     val
seed                        0       1       2       0       1       2    mean    mean
cell                                                                                 
Segment blend (G3) G3  0.6099  0.6103  0.6098  0.6552  0.6542  0.6560  0.6100  0.6551
XGBoost G1             0.5721  0.5685  0.5729  0.6467  0.6515  0.6502  0.5711  0.6495
XGBoost G3             0.5692  0.5705  0.5704  0.6402  0.6434  0.6466  0.5700  0.6434

## 6. Findings

Write this after all 24 cells are done, quoting only numbers from the tables above:

- The best model and set on test, and its margin over the next best; whether the validation ranking held (RQ2).
- The gain between models against the gain from G0 to G3 on test, with G2 → G3 as the customer-profile step (RQ3).
- How much PR-AUC changed from validation to test, and whether some models or sets dropped more than others.
- Known vs unknown customers on test, and whether the gap matches validation.
- Precision and recall at the stored threshold: what share of fraud is caught, and how many flags are false.
- The segment blend against XGBoost on G3 and G1 (section 5): whether the validation gain held on test, and on which
  customers.